## Read Neo4j credentials from Secrets Manager

In [0]:
url = dbutils.secrets.get("neo4j-aura", "NEO4J_URI")
username = dbutils.secrets.get("neo4j-aura", "NEO4J_USERNAME")
password = dbutils.secrets.get("neo4j-aura", "NEO4J_PASSWORD")
dbname = dbutils.secrets.get("neo4j-aura", "NEO4J_DATABASE")

## Create indexes in Neo4j

In [0]:
from neo4j import GraphDatabase

node_keys = [
    ('Issue', 'issue_id'),
    ('Repository', 'repository_id'),
    ('User', 'user_id'),
]

driver = GraphDatabase.driver(url, auth=(username, password))

with driver.session(database=dbname) as session:

    for label, property in node_keys:
        cypher = """CREATE CONSTRAINT IF NOT EXISTS FOR (n:{label}) REQUIRE (n.{property}) IS NODE KEY""".format(label=label, property=property)
        session.run(cypher)

## Create Spark session with Neo4j credentials

In [0]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .config("neo4j.url", url)
         .config("neo4j.authentication.basic.username", username)
         .config("neo4j.authentication.basic.password", password)
         .config("neo4j.database", dbname)
         .getOrCreate())

## Load Github issues into Spark dataframe

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import col, to_timestamp

# Define schema with timestamps as LongType (INT64 in Parquet)
schema = StructType([
    StructField("issue_id", LongType(), True),
    StructField("issue_number", LongType(), True),
    StructField("title", StringType(), True),
    StructField("body", StringType(), True),
    StructField("state", StringType(), True),
    StructField("repository_id", LongType(), True),
    StructField("repository_name", StringType(), True),
    StructField("repository_full_name", StringType(), True),
    StructField("creator_id", LongType(), True),
    StructField("creator_username", StringType(), True),
    StructField("created_at", LongType(), True),  # Read as long (nanoseconds)
    StructField("updated_at", LongType(), True),  # Read as long (nanoseconds)
    StructField("closed_at", LongType(), True),  # Read as long (nanoseconds)
    StructField("comments_count", LongType(), True),
    StructField("is_pull_request", BooleanType(), True),
    StructField("labels", StringType(), True),
    StructField("assignees", StringType(), True),
    StructField("assignee_ids", StringType(), True),
])

# Read with schema
github_issues_df = spark.read.schema(schema).parquet("/Volumes/woolford_7405614201945384/default/github-issues/").repartition(48, "issue_id")

# Convert nanosecond timestamps to TimestampType (divide by 1e9 for seconds)
for col_name in ['created_at', 'updated_at', 'closed_at']:
    github_issues_df = github_issues_df.withColumn(
        col_name,
        to_timestamp(col(col_name) / 1000000000)  # Convert nanoseconds to seconds
    )

In [0]:
display(github_issues_df)

## Create issues (only) dataframe

In [0]:
issues_df = github_issues_df.select("issue_id", "issue_number", "title", "body", "state", "created_at", "updated_at", "closed_at", "comments_count", "is_pull_request", "labels").distinct().repartition(48, "issue_id")

In [0]:
display(issues_df)

## Create repositories dataframe

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number

# 1. Define the window
window_spec = Window.partitionBy("repository_id").orderBy(col("created_at").desc())

# 2. Add row_number and 3. Filter for the first one
repository_df = github_issues_df.select(['repository_id', 'repository_name', 'repository_full_name', 'created_at']).withColumn("rn", row_number().over(window_spec)) \
             .filter("rn == 1") \
             .drop("rn", "created_at") \
             .repartition(48, "repository_id")

In [0]:
display(repository_df)

## Create users dataframe

### Get creators

In [0]:
# 1. Define the window
window_spec = Window.partitionBy("creator_id").orderBy(col("created_at").desc())

# 2. Add row_number and 3. Filter for the first one
creator_df = github_issues_df.select(['creator_id', 'creator_username', 'created_at']).withColumn("rn", row_number().over(window_spec)) \
             .filter("rn == 1") \
             .drop("rn") \
             .repartition(48)

In [0]:
display(creator_df)

### Get assignees

In [0]:
from pyspark.sql.functions import arrays_zip, split, trim, explode

# Create assignees DataFrame with both ID and username
issue_assignees_df = github_issues_df.filter(
    col("assignee_ids").isNotNull() & (trim(col("assignee_ids")) != "") &
    col("assignees").isNotNull() & (trim(col("assignees")) != "")
).select(
    col("issue_id").alias("issue_id"),
    col("created_at"),
    # Zip the two arrays together, then explode
    arrays_zip(
        split(trim(col("assignee_ids")), ","),
        split(trim(col("assignees")), ",")
    ).alias("assignee_pairs")
).select(
    col("issue_id"),
    col("created_at"),
    explode(col("assignee_pairs")).alias("pair")
).select(
    col("issue_id"),
    trim(col("pair")["0"]).cast("long").alias("assignee_id"),
    trim(col("pair")["1"]).alias("assignee_username"),
    col("created_at")
).filter(
    col("assignee_id").isNotNull() & 
    (col("assignee_username") != "") &
    (col("assignee_username") != "None")
).repartition(48)

In [0]:
display(issue_assignees_df)

### Combine creators and assignees

In [0]:
# Standardize column names and select only the columns we want to union
assignees_standardized = issue_assignees_df.select(
    col("assignee_id").alias("user_id"),
    col("assignee_username").alias("username"),
    col("created_at")
)

creators_standardized = creator_df.select(
    col("creator_id").alias("user_id"),
    col("creator_username").alias("username"),
    col("created_at")
)

# Union the two DataFrames
all_users_df = assignees_standardized.union(creators_standardized).repartition(48, "user_id")

In [0]:
# 1. Define the window
window_spec = Window.partitionBy("user_id").orderBy(col("created_at").desc())

# 2. Add row_number and 3. Filter for the first one
user_df = all_users_df.select(['user_id', 'username', 'created_at']).withColumn("rn", row_number().over(window_spec)) \
             .filter("rn == 1") \
             .drop("rn", "created_at") \
             .repartition(48, "user_id")

In [0]:
display(user_df)

## Load issues, repositories, and users

In [0]:
issues_df.write.format("org.neo4j.spark.DataSource") \
    .mode("Overwrite") \
    .option("labels", "Issue") \
    .option("node.keys", "issue_id") \
    .save()

In [0]:
repository_df.write.format("org.neo4j.spark.DataSource") \
    .mode("Overwrite") \
    .option("labels", "Repository") \
    .option("node.keys", "repository_id") \
    .save()

In [0]:
user_df.write.format("org.neo4j.spark.DataSource") \
    .mode("Overwrite") \
    .option("labels", "User") \
    .option("node.keys", "user_id") \
    .save()

## Create relationships

In [0]:
from neo4j_parallel_spark_loader import ingest_spark_dataframe
from neo4j_parallel_spark_loader.bipartite import group_and_batch_spark_dataframe

NUM_GROUPS = 12

### (:Repository)-[:HAS_ISSUE]->(:Issue)

In [0]:
repository_issue_df = github_issues_df.select(
    col("repository_id"),
    col("issue_id")
)

# Step 1: Create batches and groups for parallel ingestion
repository_issue_batched_df = group_and_batch_spark_dataframe(
    spark_dataframe=repository_issue_df,
    source_col="repository_id",
    target_col="issue_id",
    num_groups=NUM_GROUPS
)

# Step 2: Define options for Neo4j connector
options = {
    "schema.optimization.node.keys": "KEY",
    "relationship": "HAS_ISSUE",
    "relationship.save.strategy": "keys",
    "relationship.source.save.mode": "Overwrite",
    "relationship.source.labels": ":Repository",
    "relationship.source.node.keys": "repository_id:repository_id",
    "relationship.target.save.mode": "Overwrite",
    "relationship.target.labels": ":Issue",
    "relationship.target.node.keys": "issue_id:issue_id",
    # Add any relationship properties if you have them:
    "relationship.properties": "",
}

# Step 3: Ingest in parallel
ingest_spark_dataframe(
    spark_dataframe=repository_issue_batched_df,
    save_mode="Overwrite",  # or "Append"
    options=options,
    num_groups=NUM_GROUPS
)

### (:Issue)-[:ASSIGNED_TO]->(:User)

In [0]:
issue_assignees_df = issue_assignees_df.select(
    col("issue_id"),
    col("assignee_id").alias("user_id")
)

# Step 1: Create batches and groups for parallel ingestion
issue_assignees_batched_df = group_and_batch_spark_dataframe(
    spark_dataframe=issue_assignees_df,
    source_col="issue_id",
    target_col="user_id",
    num_groups=NUM_GROUPS
)

# Step 2: Define options for Neo4j connector
options = {
    "schema.optimization.node.keys": "KEY",
    "relationship": "ASSIGNED_TO",
    "relationship.save.strategy": "keys",
    "relationship.source.save.mode": "Overwrite",
    "relationship.source.labels": ":Issue",
    "relationship.source.node.keys": "issue_id:issue_id",
    "relationship.target.save.mode": "Overwrite",
    "relationship.target.labels": ":User",
    "relationship.target.node.keys": "user_id:user_id",
    # Add any relationship properties if you have them:
    "relationship.properties": "",
}

# Step 3: Ingest in parallel
ingest_spark_dataframe(
    spark_dataframe=issue_assignees_batched_df,
    save_mode="Overwrite",  # or "Append"
    options=options,
    num_groups=NUM_GROUPS
)

### (:Issue)-[:CREATED_BY]->(:User)

In [0]:
issue_creator_df = github_issues_df.select(
    col("issue_id"),
    col("creator_id").alias("user_id"),
    col("created_at"),
    col("updated_at"),
    col("closed_at")
)

# Step 1: Create batches and groups for parallel ingestion
issue_creator_batched_df = group_and_batch_spark_dataframe(
    spark_dataframe=issue_creator_df,
    source_col="issue_id",
    target_col="user_id",
    num_groups=NUM_GROUPS
)

# Step 2: Define options for Neo4j connector
options = {
    "schema.optimization.node.keys": "KEY",
    "relationship": "CREATED_BY",
    "relationship.save.strategy": "keys",
    "relationship.source.save.mode": "Overwrite",
    "relationship.source.labels": ":Issue",
    "relationship.source.node.keys": "issue_id:issue_id",
    "relationship.target.save.mode": "Overwrite",
    "relationship.target.labels": ":User",
    "relationship.target.node.keys": "user_id:user_id",
    # Add any relationship properties if you have them:
    "relationship.properties": "created_at,updated_at,closed_at",
}

# Step 3: Ingest in parallel
ingest_spark_dataframe(
    spark_dataframe=issue_creator_batched_df,
    save_mode="Overwrite",  # or "Append"
    options=options,
    num_groups=NUM_GROUPS
)